# 🏗️ Salama Insurance — Hierarchy-Based Claims RAG Agent + A/B Evaluation

**Goal:** Build a second RAG agent that uses **hierarchy-aware chunking** (preserving document section structure) and compare it head-to-head against the existing **flat-chunking** agent.

---

## Flat Chunking vs Hierarchy Chunking

| Aspect | Flat Chunking (Existing) | Hierarchy Chunking (New) |
|---|---|---|
| **Strategy** | Fixed 500-char windows with 100-char overlap across entire document | Chunk within detected section boundaries; sub-split only when a section exceeds 500 chars |
| **Section awareness** | None — chunks may span across section boundaries | Each chunk belongs to exactly one section; section name is prefixed to chunk text |
| **Element types** | All content flattened into one string | Preserves element types (title, text, table, section_header, etc.) per chunk |
| **Retrieval benefit** | Simple, uniform chunk sizes | Better semantic coherence; retriever sees "Section: Investigation Findings\n\n..." which improves relevance |
| **Trade-off** | May split mid-sentence across sections | Slightly variable chunk sizes; small sections may produce short chunks |

## Pipeline Steps
| Step | Description | Tool |
|------|-------------|------|
| 1 | Hierarchy-aware chunking from parsed elements | SQL window functions |
| 2 | Create Vector Search index on hierarchy chunks | Databricks Vector Search |
| 3 | Build & deploy hierarchy RAG agent | LangChain + MLflow |
| 4 | Build evaluation dataset (10 questions) | Manual curation |
| 5 | Evaluate both agents with MLflow | `mlflow.evaluate()` |
| 6 | UC function wrapper | SQL `ai_query()` |

**Source data:** `salama_insurance.salama_silver.claim_documents_parsed` (25 PDFs already parsed via `ai_parse_document()`)

In [0]:
%pip install -U -qqqq databricks-agents>=0.16.0 "mlflow[databricks]>=2.20.2" databricks-langchain databricks-vectorsearch langchain langchain-community

In [0]:
dbutils.library.restartPython()

In [0]:
# ---------------------------------------------------------------------------
# Configuration — Hierarchy Agent + Evaluation
# ---------------------------------------------------------------------------
CATALOG = "salama_insurance"
SCHEMA = "salama_silver"

# Reuse existing parsed documents
PARSED_TABLE = f"{CATALOG}.{SCHEMA}.claim_documents_parsed"

# --- Hierarchy agent assets (NEW) ---
HIERARCHY_CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.claim_document_chunks_hierarchy"
HIERARCHY_VS_INDEX = f"{CATALOG}.{SCHEMA}.claim_documents_index_hierarchy"
HIERARCHY_AGENT_NAME = "claims_rag_agent_hierarchy"
HIERARCHY_UC_MODEL = f"{CATALOG}.{SCHEMA}.claims_rag_agent_hierarchy"

# --- Existing flat agent assets (for comparison) ---
FLAT_CHUNKS_TABLE = f"{CATALOG}.{SCHEMA}.claim_document_chunks"
FLAT_VS_INDEX = f"{CATALOG}.{SCHEMA}.claim_documents_index"
FLAT_AGENT_ENDPOINT = "claims_rag_agent"

# --- Shared resources ---
VS_ENDPOINT = "claims_documents_vs_endpoint"
LLM_ENDPOINT = "databricks-claude-sonnet-4-5"
EMBEDDING_MODEL = "databricks-bge-large-en"

print("✅ Configuration loaded")
print(f"   Parsed docs:       {PARSED_TABLE}")
print(f"   Hierarchy chunks:  {HIERARCHY_CHUNKS_TABLE}")
print(f"   Hierarchy index:   {HIERARCHY_VS_INDEX}")
print(f"   Hierarchy agent:   {HIERARCHY_AGENT_NAME}")
print(f"   Flat agent:        {FLAT_AGENT_ENDPOINT}")
print(f"   LLM:               {LLM_ENDPOINT}")

In [0]:
# Hierarchy-aware chunking - respects document section boundaries
result = spark.sql("""
CREATE OR REPLACE TABLE salama_insurance.salama_silver.claim_document_chunks_hierarchy AS
WITH
elements_exploded AS (
  SELECT
    document_name,
    document_type,
    path AS source_uri,
    pos AS element_pos,
    try_cast(element:type AS STRING) AS element_type,
    try_cast(element:content AS STRING) AS element_content
  FROM salama_insurance.salama_silver.claim_documents_parsed
    LATERAL VIEW posexplode(
      try_cast(parsed:document:elements AS ARRAY<VARIANT>)
    ) AS pos, element
  WHERE full_text IS NOT NULL AND LENGTH(full_text) > 0
),
section_assigned AS (
  SELECT
    *,
    COALESCE(
      LAST_VALUE(
        CASE
          WHEN element_type IN ('section_header', 'title') THEN element_content
          ELSE NULL
        END, true
      ) OVER (
        PARTITION BY document_name
        ORDER BY element_pos
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
      ),
      'Document Header'
    ) AS section_name
  FROM elements_exploded
),
section_content AS (
  SELECT
    document_name,
    document_type,
    source_uri,
    section_name,
    collect_list(element_type) AS element_types,
    concat_ws('\\n', collect_list(
      CASE WHEN element_type NOT IN ('page_footer', 'page_header', 'page_number', 'footnote')
           THEN element_content END
    )) AS section_text,
    MIN(element_pos) AS section_start_pos
  FROM section_assigned
  WHERE element_content IS NOT NULL AND LENGTH(TRIM(element_content)) > 0
  GROUP BY document_name, document_type, source_uri, section_name
),
sub_chunked AS (
  SELECT
    document_name,
    document_type,
    source_uri,
    section_name,
    element_types,
    section_start_pos,
    posexplode(
      transform(
        sequence(0, greatest(length(section_text) - 1, 0), 400),
        pos -> substring(section_text, pos + 1, 500)
      )
    ) AS (chunk_pos, raw_chunk_text)
  FROM section_content
  WHERE LENGTH(TRIM(section_text)) > 0
)
SELECT
  md5(concat(document_name, '_', section_name, '_', cast(chunk_pos AS STRING))) AS chunk_id,
  document_name,
  document_type,
  section_name,
  element_types,
  concat('Section: ', section_name, '\\n\\n', raw_chunk_text) AS chunk_text,
  source_uri,
  chunk_pos,
  current_timestamp() AS indexed_at
FROM sub_chunked
WHERE LENGTH(TRIM(raw_chunk_text)) > 50
""")

print("✅ Hierarchy chunks table created!")
count = spark.sql("SELECT COUNT(*) AS cnt FROM salama_insurance.salama_silver.claim_document_chunks_hierarchy").collect()[0][0]
print(f"   Total hierarchy chunks: {count}")

In [0]:
# Compare hierarchy chunks vs flat chunks side-by-side
comparison_df = spark.sql("""
WITH hierarchy_stats AS (
  SELECT
    'Hierarchy' AS strategy,
    COUNT(*) AS total_chunks,
    COUNT(DISTINCT section_name) AS distinct_sections,
    ROUND(AVG(LENGTH(chunk_text)), 0) AS avg_chunk_len,
    MIN(LENGTH(chunk_text)) AS min_chunk_len,
    MAX(LENGTH(chunk_text)) AS max_chunk_len
  FROM salama_insurance.salama_silver.claim_document_chunks_hierarchy
),
flat_stats AS (
  SELECT
    'Flat' AS strategy,
    COUNT(*) AS total_chunks,
    0 AS distinct_sections,
    ROUND(AVG(LENGTH(chunk_text)), 0) AS avg_chunk_len,
    MIN(LENGTH(chunk_text)) AS min_chunk_len,
    MAX(LENGTH(chunk_text)) AS max_chunk_len
  FROM salama_insurance.salama_silver.claim_document_chunks
)
SELECT * FROM hierarchy_stats
UNION ALL
SELECT * FROM flat_stats
""")

print("📊 Chunking Strategy Comparison:")
display(comparison_df)

# Show section name distribution
print("\n🏷️ Distinct section names in hierarchy chunks:")
sections_df = spark.sql("""
SELECT section_name, COUNT(*) AS chunk_count
FROM salama_insurance.salama_silver.claim_document_chunks_hierarchy
GROUP BY section_name
ORDER BY chunk_count DESC
""")
display(sections_df)

In [0]:
from databricks.vector_search.client import VectorSearchClient

vsc = VectorSearchClient()

# --- Verify endpoint exists (reuse the same one) ---
try:
    ep = vsc.get_endpoint(VS_ENDPOINT)
    print(f"✅ Endpoint '{VS_ENDPOINT}' already exists (status: {ep.get('endpoint_status', {}).get('state', 'unknown')})")
except Exception as e:
    print(f"⚠️ Endpoint not found. Creating '{VS_ENDPOINT}'...")
    vsc.create_endpoint(name=VS_ENDPOINT, endpoint_type="STANDARD")
    print(f"✅ Endpoint '{VS_ENDPOINT}' creation initiated.")

# --- Delete existing hierarchy index if it exists (for clean rebuild) ---
try:
    vsc.get_index(VS_ENDPOINT, HIERARCHY_VS_INDEX)
    print(f"♻️ Deleting existing hierarchy index for clean rebuild...")
    vsc.delete_index(VS_ENDPOINT, HIERARCHY_VS_INDEX)
    import time; time.sleep(5)
except:
    pass

# --- Create Delta Sync index on hierarchy chunks ---
print(f"\n🔨 Creating Delta Sync index: {HIERARCHY_VS_INDEX}")
print(f"   Source table: {HIERARCHY_CHUNKS_TABLE}")
print(f"   Embedding model: {EMBEDDING_MODEL}")

index = vsc.create_delta_sync_index(
    endpoint_name=VS_ENDPOINT,
    index_name=HIERARCHY_VS_INDEX,
    source_table_name=HIERARCHY_CHUNKS_TABLE,
    pipeline_type="TRIGGERED",
    primary_key="chunk_id",
    embedding_source_column="chunk_text",
    embedding_model_endpoint_name=EMBEDDING_MODEL,
    columns_to_sync=["chunk_id", "document_name", "document_type", "section_name", "chunk_text"],
)

print(f"✅ Index creation initiated: {HIERARCHY_VS_INDEX}")
print("   Run the next cell to wait for it to become ready.")

In [0]:
import time
from databricks.sdk import WorkspaceClient

w_vs = WorkspaceClient()

# --- Wait for hierarchy index to be ready (up to 25 minutes) ---
print("Waiting for hierarchy index to be ready (this can take 15-20 min)...")
for i in range(150):  # 150 x 10s = 25 minutes
    try:
        idx_status = w_vs.vector_search_indexes.get_index(HIERARCHY_VS_INDEX)
        is_ready = idx_status.status.ready if idx_status.status else False
        msg = idx_status.status.message if idx_status.status else "unknown"
        if is_ready:
            print(f"\n✅ Hierarchy index is ready! (took ~{i * 10}s)")
            break
        if i % 6 == 0:  # print every 60s
            print(f"   ⏳ {msg} ({i * 10}s elapsed)")
    except Exception as e:
        if i % 6 == 0:
            print(f"   ⏳ Waiting... {str(e)[:80]} ({i * 10}s elapsed)")
    time.sleep(10)
else:
    print("⚠️ Index not ready after 25 minutes. Check VS UI for status.")
    print("   Re-run this cell later to continue.")
    import sys; sys.exit(0)

# --- Test retrieval: compare hierarchy vs flat ---
from databricks.vector_search.client import VectorSearchClient
vsc = VectorSearchClient(disable_notice=True)

# Discover the correct endpoint for the flat index
flat_idx_info = w_vs.vector_search_indexes.get_index(FLAT_VS_INDEX)
FLAT_VS_ENDPOINT = flat_idx_info.endpoint_name
print(f"\nFlat index endpoint: {FLAT_VS_ENDPOINT}")
print(f"Hierarchy index endpoint: {VS_ENDPOINT}")

test_query = "What are the fraud investigation findings?"

print(f"\n{'='*70}")
print(f"Test query: '{test_query}'")
print(f"{'='*70}")

# Hierarchy index results
hierarchy_index = vsc.get_index(VS_ENDPOINT, HIERARCHY_VS_INDEX)
hierarchy_results = hierarchy_index.similarity_search(
    query_text=test_query,
    columns=["document_name", "document_type", "section_name", "chunk_text"],
    num_results=3,
)

print("\n🏗️ HIERARCHY CHUNKS (top 3):")
print("-" * 40)
for row in hierarchy_results.get("result", {}).get("data_array", []):
    print(f"  Doc: {row[0]} | Type: {row[1]} | Section: {row[2]}")
    print(f"  Preview: {row[3][:120]}...")
    print()

# Flat index results — use the correct endpoint
flat_index = vsc.get_index(FLAT_VS_ENDPOINT, FLAT_VS_INDEX)
flat_results = flat_index.similarity_search(
    query_text=test_query,
    columns=["document_name", "document_type", "chunk_text"],
    num_results=3,
)

print("📏 FLAT CHUNKS (top 3):")
print("-" * 40)
for row in flat_results.get("result", {}).get("data_array", []):
    print(f"  Doc: {row[0]} | Type: {row[1]}")
    print(f"  Preview: {row[2][:120]}...")
    print()

In [0]:
import mlflow
from databricks_langchain import ChatDatabricks, VectorSearchRetrieverTool
from langchain.agents import create_agent

mlflow.langchain.autolog()

# --- Initialize retriever tool pointing to hierarchy index ---
claims_hierarchy_retriever = VectorSearchRetrieverTool(
    index_name=HIERARCHY_VS_INDEX,
    tool_name="claims_hierarchy_document_search",
    tool_description=(
        "Searches Salama Insurance claim documents using hierarchy-aware chunks. "
        "Each chunk preserves its document section context (e.g., Policyholder Details, "
        "Claim Summary, Investigation Findings, Settlement Terms). "
        "Use this tool to find information about specific claims, policy details, "
        "claimed amounts, settlement amounts, fraud investigations, denial reasons, "
        "and customer information. Results include the section name for precise attribution."
    ),
    num_results=5,
    columns=["chunk_id", "document_name", "document_type", "section_name", "chunk_text"],
)

# --- LLM ---
llm = ChatDatabricks(endpoint=LLM_ENDPOINT, temperature=0.1)

# --- System prompt emphasizing section-aware citations ---
HIERARCHY_SYSTEM_PROMPT = """You are a Salama Insurance Claims Assistant AI agent (Hierarchy-Enhanced).
You help insurance staff search and analyze claim documents including:
- Claim Submission Forms (policyholder info, policy details, claimed amounts)
- Settlement Notifications (approved amounts, payment details)
- Investigation Reports (fraud scores, investigation findings)
- Denial Letters (rejection reasons, appeal process)

IMPORTANT CITATION RULES:
1. Always cite the DOCUMENT NAME and SECTION NAME when providing information.
   Example: "According to claim_form_CLM000146 (Section: Claim Summary), the claimed amount is..."
2. When information comes from multiple sections, organize your response by section.
3. If you cannot find the answer in the retrieved documents, say so clearly.
4. Format financial amounts with AED currency and proper formatting.
5. When summarizing across documents, list each source document and section."""

# --- Build agent ---
agent = create_agent(
    model=llm,
    tools=[claims_hierarchy_retriever],
    system_prompt=HIERARCHY_SYSTEM_PROMPT,
)

print("✅ Hierarchy Claims RAG Agent ready!")
print(f"   LLM: {LLM_ENDPOINT}")
print(f"   Retriever index: {HIERARCHY_VS_INDEX}")
print(f"   Tool: claims_hierarchy_document_search")

# --- Quick test ---
print("\n🧪 Testing agent with a sample question...")
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What claims were denied and what were the reasons?"}]}
)
final_msg = [m for m in response["messages"] if hasattr(m, "content") and m.type == "ai" and len(m.content) > 50][-1]
print(f"\n💬 Agent response preview:\n{final_msg.content[:500]}...")

In [0]:
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import EndpointCoreConfigInput, ServedEntityInput

w = WorkspaceClient()

# Model v3 logged with resources (DatabricksVectorSearchIndex + DatabricksServingEndpoint)
HIERARCHY_UC_MODEL = "salama_insurance.salama_silver.claims_rag_agent_hierarchy"
MODEL_VERSION = "3"  # Version with resource dependencies

print(f"✅ Using model: {HIERARCHY_UC_MODEL} v{MODEL_VERSION} (with resource deps)")

# ---------------------------------------------------------------------------
# Deploy via SDK (agents.deploy rejects [Any] output schema in newer versions)
# ---------------------------------------------------------------------------
print(f"\n🚀 Creating endpoint: {HIERARCHY_AGENT_NAME}")

try:
    w.serving_endpoints.create(
        name=HIERARCHY_AGENT_NAME,
        config=EndpointCoreConfigInput(
            name=HIERARCHY_AGENT_NAME,
            served_entities=[
                ServedEntityInput(
                    entity_name=HIERARCHY_UC_MODEL,
                    entity_version=MODEL_VERSION,
                    scale_to_zero_enabled=True,
                    workload_size="Small",
                )
            ],
        ),
    )
    print("✅ Endpoint creation initiated (10-15 min)...")
except Exception as e:
    if "already exists" in str(e).lower() or "already being" in str(e).lower():
        print(f"   Endpoint exists, updating config...")
        w.serving_endpoints.update_config(
            name=HIERARCHY_AGENT_NAME,
            served_entities=[
                ServedEntityInput(
                    entity_name=HIERARCHY_UC_MODEL,
                    entity_version=MODEL_VERSION,
                    scale_to_zero_enabled=True,
                    workload_size="Small",
                )
            ],
        )
        print("✅ Update initiated...")
    else:
        raise

# Monitor deployment
for i in range(90):
    try:
        ep = w.serving_endpoints.get(HIERARCHY_AGENT_NAME)
        state = ep.state
        ready = state.ready.value if state and state.ready else "UNKNOWN"
        config_update = state.config_update.value if state and state.config_update else "UNKNOWN"
        
        if ready == "READY":
            print(f"\n✅ Endpoint is READY! (took ~{i * 10}s)")
            break
        if config_update == "UPDATE_FAILED":
            if ep.pending_config and ep.pending_config.served_entities:
                for e in ep.pending_config.served_entities:
                    print(f"\n❌ FAILED: {e.state.deployment_state_message}")
            break
        if i % 6 == 0:
            print(f"   ⏳ Ready={ready}, Config={config_update} ({i * 10}s)")
    except Exception as e:
        if i % 6 == 0:
            print(f"   ⏳ Waiting... ({i * 10}s)")
    time.sleep(10)
else:
    print(f"⚠️ Not ready after 15 min. Check /serving-endpoints/{HIERARCHY_AGENT_NAME}")

In [0]:
import pandas as pd

# ---------------------------------------------------------------------------
# Evaluation dataset: 10 representative questions across different categories
# These questions test retrieval quality, answer accuracy, and citation behavior
# ---------------------------------------------------------------------------

eval_data = [
    # --- Specific claim lookups ---
    {
        "request": "What is the claimed amount for claim CLM000146?",
        "expected_facts": ["CLM000146", "claimed amount", "AED"],
        "category": "specific_lookup",
    },
    {
        "request": "Who is the policyholder for claim CLM001461 and what is their policy number?",
        "expected_facts": ["CLM001461", "policyholder", "policy number"],
        "category": "specific_lookup",
    },
    # --- Cross-document questions ---
    {
        "request": "Which claims were denied and what were the reasons for denial?",
        "expected_facts": ["denial", "reason", "denied"],
        "category": "cross_document",
    },
    {
        "request": "List all settlement notifications and their approved amounts.",
        "expected_facts": ["settlement", "approved", "amount", "AED"],
        "category": "cross_document",
    },
    # --- Section-specific questions (hierarchy advantage) ---
    {
        "request": "What are the investigation findings for fraud cases? Include fraud scores.",
        "expected_facts": ["investigation", "fraud", "score", "findings"],
        "category": "section_specific",
    },
    {
        "request": "What policyholder details are available for claims with investigation reports?",
        "expected_facts": ["policyholder", "name", "investigation"],
        "category": "section_specific",
    },
    {
        "request": "Summarize the claim summary section across all claim forms.",
        "expected_facts": ["claim summary", "incident", "description"],
        "category": "section_specific",
    },
    # --- Financial summaries ---
    {
        "request": "What is the total claimed amount across all claim forms?",
        "expected_facts": ["total", "AED", "claimed"],
        "category": "financial_summary",
    },
    {
        "request": "Compare the claimed amounts vs settlement amounts where both are available.",
        "expected_facts": ["claimed", "settlement", "AED"],
        "category": "financial_summary",
    },
    # --- Complex reasoning ---
    {
        "request": "Which claims have the highest fraud risk based on investigation reports, and what evidence supports this?",
        "expected_facts": ["fraud", "risk", "evidence", "investigation"],
        "category": "complex_reasoning",
    },
]

eval_df = pd.DataFrame(eval_data)
print(f"✅ Evaluation dataset created: {len(eval_df)} questions")
print(f"\nCategory distribution:")
print(eval_df["category"].value_counts().to_string())
print(f"\nSample questions:")
for i, row in eval_df.head(3).iterrows():
    print(f"  [{row['category']}] {row['request']}")

# Convert to MLflow evaluation format
eval_dataset = eval_df[["request"]].copy()
display(eval_dataset)

In [0]:
import mlflow
import pandas as pd
import time
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

# ---------------------------------------------------------------------------
# Helper: Query a serving endpoint
# ---------------------------------------------------------------------------
def query_agent_endpoint(endpoint_name, question, timeout=120):
    """Query an agent serving endpoint and return the response."""
    try:
        response = w.serving_endpoints.query(
            name=endpoint_name,
            messages=[{"role": "user", "content": question}],
        )
        if hasattr(response, "choices") and response.choices:
            return response.choices[0].message.content
        return str(response)
    except Exception as e:
        return f"Error querying {endpoint_name}: {str(e)[:200]}"

# ---------------------------------------------------------------------------
# Wait for both endpoints
# ---------------------------------------------------------------------------
print("Checking endpoint readiness...")
for name in [FLAT_AGENT_ENDPOINT, HIERARCHY_AGENT_NAME]:
    for attempt in range(30):
        try:
            ep = w.serving_endpoints.get(name)
            state = ep.state.ready.value if ep.state and ep.state.ready else "UNKNOWN"
            if state == "READY":
                print(f"  ✅ {name}: READY")
                break
            print(f"  ⏳ {name}: {state} (attempt {attempt + 1}/30)")
            time.sleep(20)
        except Exception as e:
            print(f"  ⚠️ {name}: {e}")
            break
    else:
        print(f"  ⚠️ {name}: Not ready after 10 minutes")

# ---------------------------------------------------------------------------
# Evaluate both agents
# ---------------------------------------------------------------------------
def evaluate_agent(agent_name, endpoint_name, strategy_name, chunking_desc):
    """Evaluate a single agent and return metrics."""
    print(f"\n{'='*70}")
    print(f"Evaluating: {agent_name} ({endpoint_name})")
    print(f"{'='*70}")
    
    def agent_model(inputs):
        results = []
        for idx, row in inputs.iterrows():
            print(f"  Q{idx+1}: {row['request'][:60]}...")
            answer = query_agent_endpoint(endpoint_name, row["request"])
            results.append(answer)
        return results
    
    with mlflow.start_run(run_name=f"eval_{strategy_name}") as run:
        mlflow.log_param("agent_type", strategy_name)
        mlflow.log_param("endpoint", endpoint_name)
        mlflow.log_param("chunking_strategy", chunking_desc)
        
        eval_results = mlflow.evaluate(
            model=agent_model,
            data=eval_dataset,
            model_type="databricks-agent",
        )
        
        print(f"\nMetrics for {agent_name}:")
        for k, v in sorted(eval_results.metrics.items()):
            if isinstance(v, float):
                print(f"  {k}: {v:.4f}")
            else:
                print(f"  {k}: {v}")
        
        return eval_results, run.info.run_id

flat_results, flat_run_id = evaluate_agent(
    "Flat Agent", FLAT_AGENT_ENDPOINT, "flat_chunking", "fixed_500char_100overlap"
)
hierarchy_results, hier_run_id = evaluate_agent(
    "Hierarchy Agent", HIERARCHY_AGENT_NAME, "hierarchy_chunking", "section_aware_500char_100overlap"
)

# ---------------------------------------------------------------------------
# Side-by-side comparison
# ---------------------------------------------------------------------------
print(f"\n{'='*70}")
print("SIDE-BY-SIDE COMPARISON")
print(f"{'='*70}")

all_metrics = set(flat_results.metrics.keys()) | set(hierarchy_results.metrics.keys())
numeric_metrics = []
for m in sorted(all_metrics):
    flat_val = flat_results.metrics.get(m)
    hier_val = hierarchy_results.metrics.get(m)
    if isinstance(flat_val, (int, float)) and isinstance(hier_val, (int, float)):
        diff = hier_val - flat_val
        winner = "⬆️ Hierarchy" if diff > 0.001 else ("⬇️ Flat" if diff < -0.001 else "➖ Tie")
        numeric_metrics.append({
            "Metric": m,
            "Flat Agent": round(flat_val, 4),
            "Hierarchy Agent": round(hier_val, 4),
            "Difference": round(diff, 4),
            "Winner": winner,
        })

if numeric_metrics:
    comparison_df = pd.DataFrame(numeric_metrics)
    display(comparison_df)
    
    hier_wins = sum(1 for r in numeric_metrics if "Hierarchy" in r["Winner"])
    flat_wins = sum(1 for r in numeric_metrics if "Flat" in r["Winner"])
    ties = sum(1 for r in numeric_metrics if "Tie" in r["Winner"])
    print(f"\n🏆 Summary: Hierarchy wins {hier_wins} | Flat wins {flat_wins} | Ties {ties}")
else:
    print("\n⚠️ No numeric metrics to compare. Check evaluation results above.")
    print("Flat metrics:", flat_results.metrics)
    print("Hierarchy metrics:", hierarchy_results.metrics)

print(f"\n   Flat eval run:      {flat_run_id}")
print(f"   Hierarchy eval run: {hier_run_id}")

In [0]:
# Create Unity Catalog function that wraps the hierarchy claims_rag_agent_hierarchy endpoint
# Allows anyone to call: SELECT salama_insurance.salama_silver.ask_claims_agent_hierarchy('your question')

spark.sql("""
CREATE OR REPLACE FUNCTION salama_insurance.salama_silver.ask_claims_agent_hierarchy(
  question STRING
  COMMENT 'Natural language question about claim documents (uses hierarchy-aware chunking)'
)
RETURNS STRING
COMMENT 'Searches Salama Insurance claim documents using the Hierarchy RAG Agent. Uses section-aware chunking for better retrieval of document sections like Policyholder Details, Claim Summary, Investigation Findings. Returns AI-generated answers citing document names, section names, and financial amounts in AED.'
RETURN (
  SELECT element_at(
    filter(
      ai_query('claims_rag_agent_hierarchy',
        request => named_struct(
          'messages', array(
            named_struct('role', 'user', 'content', question)
          )
        ),
        returnType => 'STRUCT<messages:ARRAY<STRUCT<content:STRING, type:STRING>>>'
      ).messages,
      m -> m.type = 'ai' AND m.content IS NOT NULL AND length(m.content) > 50
    ),
    -1
  ).content
)
""")
print("✅ UC function created: salama_insurance.salama_silver.ask_claims_agent_hierarchy")

# Quick test
print("\n🧪 Testing UC function...")
result = spark.sql("""
SELECT salama_insurance.salama_silver.ask_claims_agent_hierarchy(
  'What are the fraud investigation findings?'
) AS hierarchy_agent_response
""")
display(result)

# 📊 Summary & Recommendations

## What We Built
1. **Hierarchy-aware chunking pipeline** that preserves document section structure from `ai_parse_document()` elements
2. **Dedicated Vector Search index** (`claim_documents_index_hierarchy`) with section-prefixed chunks
3. **Hierarchy RAG agent** (`claims_rag_agent_hierarchy`) with section-aware system prompt
4. **A/B evaluation harness** comparing both agents across 10 representative questions using `mlflow.evaluate()`
5. **UC function wrapper** (`ask_claims_agent_hierarchy`) for SQL-based access

## Key Differences in Chunking

| Feature | Flat Agent | Hierarchy Agent |
|---------|-----------|----------------|
| Chunk boundaries | Fixed 500-char windows | Section-aware boundaries |
| Context prefix | None | "Section: {name}\n\n" prepended |
| Element filtering | None | Excludes page_footer, page_header, page_number, footnote |
| Section metadata | Not available | `section_name` column for filtering & attribution |
| Citation quality | Document-level only | Document + Section level |

## When to Use Which

**Prefer Hierarchy Agent when:**
- Questions target specific document sections ("What are the investigation findings?")
- Citation precision matters (regulatory, audit scenarios)
- Documents have clear section structure (forms, reports, legal letters)

**Prefer Flat Agent when:**
- Questions span entire documents without section focus
- Simplicity and uniform chunk sizes are preferred
- Documents lack clear section structure

## Next Steps
- Review the MLflow evaluation comparison table above
- Check per-question results in the MLflow experiment UI for detailed breakdowns
- Consider A/B testing in production by routing a percentage of traffic to each endpoint
- Use the UC functions (`ask_claims_agent` vs `ask_claims_agent_hierarchy`) from dashboards or SQL queries

---
*Both agents are accessible via:*
- **Flat:** `SELECT salama_insurance.salama_silver.ask_claims_agent('question')`
- **Hierarchy:** `SELECT salama_insurance.salama_silver.ask_claims_agent_hierarchy('question')`